# Multi-Label Emotion Classification with IndicBERT (Enhanced Reporting)

This notebook trains **IndicBERT** (`ai4bharat/indic-bert`) for multi-label classification of Indian poetry.

## key Features
- **Model**: `ai4bharat/indic-bert`
- **Task**: Multi-Label Classification (Primary + Secondary).
- **Reporting**: Learning Curves, Weighted Metrics, Class-wise Report.

In [ ]:
!pip install transformers accelerate scikit-learn openpyxl sentencepiece protobuf huggingface_hub -q

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Output directory for reports
reports_dir = 'round2/reports'
os.makedirs(reports_dir, exist_ok=True)

In [ ]:
# --- 1. Load Data ---

# Fixed Absolute Path as per project standard
FILE_PATH = '/Users/hemishjain22/Desktop/Hackathons/hack4healtj/round2/Combined_Emotions.xlsx'
print(f"Loading: {FILE_PATH}")

if os.path.exists(FILE_PATH):
    df = pd.read_excel(FILE_PATH)
    df['Poem'] = df['Poem'].fillna('')
    df['Primary'] = df['Primary'].fillna('')
    df['Secondary'] = df['Secondary'].fillna('')
    print(f"Loaded {len(df)} rows.")
else:
    # Fallback for Kaggle/Other envs if needed, or error
    raise FileNotFoundError(f"{FILE_PATH} not found.")

In [ ]:
# --- 2. Label Encoding (Multi-Label) ---
primary_set = set(df['Primary'].unique()) - {''}
secondary_set = set(df['Secondary'].unique()) - {''}
all_labels = sorted(list(primary_set.union(secondary_set)))

label_map = {label: i for i, label in enumerate(all_labels)}
inv_map = {i: label for label, i in label_map.items()}
num_classes = len(all_labels)
print(f"Unique Labels: {num_classes}")

def create_targets(df, label_map):
    targets = []
    for _, row in df.iterrows():
        vec = np.zeros(len(label_map), dtype=np.float32)
        # Multi-hot encoding
        if row['Primary'] in label_map:
            vec[label_map[row['Primary']]] = 1.0
        if row['Secondary'] in label_map:
            vec[label_map[row['Secondary']]] = 1.0
        targets.append(vec)
    return np.array(targets)

y_all = create_targets(df, label_map)
X_all = df['Poem'].values

# 80/20 Split
X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
# --- 3. Dataset & Tokenizer ---
MODEL_NAME = 'ai4bharat/indic-bert'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, keep_accents=True)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds = TextDataset(X_val, y_val, tokenizer)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=16)

In [ ]:
# --- 4. Model Setup ---
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    problem_type="multi_label_classification"
)
model = model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
# --- 5. Training Loop with History ---

history = {
    'train_loss': [],
    'val_loss': [],
    'val_f1_micro': []
}

def train_step(model, dl):
    model.train()
    total_loss = 0
    for batch in tqdm(dl, desc="Train"):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        
        optimizer.zero_grad()
        output = model(input_ids, attention_mask=mask, labels=lbls)
        loss = output.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dl)

def eval_step(model, dl):
    model.eval()
    total_loss = 0
    all_preds = []
    all_true = []
    
    with torch.no_grad():
        for batch in dl:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            
            output = model(input_ids, attention_mask=mask, labels=lbls)
            total_loss += output.loss.item()
            
            # Sigmoid for multi-label
            probs = torch.sigmoid(output.logits).cpu().numpy()
            all_preds.extend(probs)
            all_true.extend(lbls.cpu().numpy())
            
    preds_bin = (np.array(all_preds) > 0.5).astype(int)
    true_bin = np.array(all_true).astype(int)
    
    # Standard metric for tracking
    f1_mic = f1_score(true_bin, preds_bin, average='micro', zero_division=0)
    return total_loss / len(dl), f1_mic

EPOCHS = 10
best_score = 0
SAVE_PATH = 'round2/indicbert_model'
os.makedirs(SAVE_PATH, exist_ok=True)

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    t_loss = train_step(model, train_dl)
    v_loss, f1_mic = eval_step(model, val_dl)
    
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['val_f1_micro'].append(f1_mic)
    
    print(f"Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Micro F1: {f1_mic:.4f}")
    
    if f1_mic > best_score:
        best_score = f1_mic
        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)
        print("\033[92mSaved Best Model!\033[0m")

In [ ]:
# --- 6. Learning Curve (Train vs Val) ---
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS+1), history['train_loss'], 'o-', label='Training Loss')
plt.plot(range(1, EPOCHS+1), history['val_loss'], 'o-', label='Validation Loss')
plt.title('IndicBERT Learning Curve (Loss)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.savefig(f'{reports_dir}/indicbert_learning_curve.png')
plt.show()

In [ ]:
# --- 7. Final Evaluation (Weighted Metrics) ---
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report


# Load Best Model
print("Loading best model for evaluation...")
model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
model = model.to(device)
model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for batch in tqdm(val_dl, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        output = model(input_ids, attention_mask=mask)
        probs = torch.sigmoid(output.logits).cpu().numpy()
        all_preds.extend(probs)
        all_true.extend(lbls.cpu().numpy())

y_true = np.array(all_true).astype(int)
y_pred = (np.array(all_preds) > 0.5).astype(int)

# Metrics
acc = accuracy_score(y_true, y_pred)
f1_w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
prec_w = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec_w = recall_score(y_true, y_pred, average='weighted', zero_division=0)

print("\n--- IndicBERT Multi-Label Report ---")
print(f"Accuracy (Full Match): {acc:.4f}")
print(f"Weighted Precision: {prec_w:.4f}")
print(f"Weighted Recall: {rec_w:.4f}")
print(f"Weighted F1: {f1_w:.4f}")

# Classification Report

# Ensure inv_map is defined (if cell run independently)
if 'inv_map' not in locals() and 'label_map' in locals():
    inv_map = {i: label for label, i in label_map.items()}

target_names = [inv_map[i] for i in range(len(inv_map))]
report = classification_report(y_true, y_pred, target_names=target_names, zero_division=0, output_dict=True)
pd.DataFrame(report).transpose().to_csv(f'{reports_dir}/indicbert_class_report.csv')
print("Class-wise report saved.")